In [2]:
import sys
sys.path.append("..")
import json
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score
from src.features import build_features, MODEL_B_DROP

In [3]:
train = pd.read_parquet("../data/processed/train.parquet")
test = pd.read_parquet("../data/processed/test.parquet")
stress = pd.read_parquet("../data/processed/stress.parquet")

In [4]:
model = XGBClassifier()
model.load_model("../models/xgb_model_b.json")
thresholds = json.load(open("../models/thresholds.json"))

In [5]:
def score(d):
    X = build_features(d)
    feats = [c for c in X.columns if c not in MODEL_B_DROP]
    return model.predict_proba(X[feats])[:, 1]

test_scores = score(test)
stress_scores = score(stress)

In [6]:
def summary(d, s, t, review_cost=1000):
    y = d["isFraud"].values
    amt = d["amount"].values
    alert = s >= t
    caught = (y == 1) & alert
    missed = amt[(y == 1) & ~alert].sum()
    return {
        "alerts": int(alert.sum()),
        "alerts_per_hour": round(alert.sum() / d["step"].nunique(), 1),
        "frauds": int(y.sum()),
        "caught": int(caught.sum()),
        "precision": round(caught.sum() / max(alert.sum(), 1), 3),
        "recall": round(caught.sum() / y.sum(), 3),
        "missed_loss": round(missed),
        "total_cost": round(missed + review_cost * alert.sum()),
        "no_alert_cost": round(amt[y == 1].sum()),
    }

y_test = test["isFraud"].values
print("Test PR-AUC:", round(average_precision_score(y_test, test_scores), 4))

rows = []
for name, t in thresholds.items():
    rows.append({"setting": name, "threshold": t, **summary(test, test_scores, t)})
rows.append({"setting": "rule: isFlaggedFraud", "threshold": None,
             **summary(test, test["isFlaggedFraud"].values.astype(float), 0.5)})
pd.DataFrame(rows)

Test PR-AUC: 0.7486


,setting,threshold,alerts,alerts_per_hour,frauds,caught,precision,recall,missed_loss,total_cost,no_alert_cost
0,main_cost_1000,0.0100,6178,121.1,590,553,0.09,0.937,4093440,10271440,1079181691
1,capacity_20_per_hour,0.0565,1135,22.3,590,465,0.41,0.788,17215800,18350800,1079181691
2,rule: isFlaggedFraud,NaN,1,0.0,590,1,1.00,0.002,1074289498,1074290498,1079181691


In [ ]:
def summary(d, s, t, review_cost=1000):
    y = d["isFraud"].values
    amt = d["amount"].values
    alert = s >= t
    caught = (y == 1) & alert
    missed = amt[(y == 1) & ~alert].sum()
    return {
        "alerts": int(alert.sum()),
        "alerts_per_hour": round(alert.sum() / d["step"].nunique(), 1),
        "frauds": int(y.sum()),
        "caught": int(caught.sum()),
        "precision": round(caught.sum() / max(alert.sum(), 1), 3),
        "recall": round(caught.sum() / y.sum(), 3),
        "missed_loss": round(missed),
        "total_cost": round(missed + review_cost * alert.sum()),
        "no_alert_cost": round(amt[y == 1].sum()),
    }

y_test = test["isFraud"].values
print("Test PR-AUC:", round(average_precision_score(y_test, test_scores), 4))

rows = []
for name, t in thresholds.items():
    rows.append({"setting": name, "threshold": t, **summary(test, test_scores, t)})
rows.append({"setting": "rule: isFlaggedFraud", "threshold": None,
             **summary(test, test["isFlaggedFraud"].values.astype(float), 0.5)})
pd.DataFrame(rows)

Test PR-AUC: 0.7486


,setting,threshold,alerts,alerts_per_hour,frauds,caught,precision,recall,missed_loss,total_cost,no_alert_cost
0,main_cost_1000,0.0100,6178,121.1,590,553,0.09,0.937,4093440,10271440,1079181691
1,capacity_20_per_hour,0.0565,1135,22.3,590,465,0.41,0.788,17215800,18350800,1079181691
2,rule: isFlaggedFraud,NaN,1,0.0,590,1,1.00,0.002,1074289498,1074290498,1079181691


In [7]:
t = thresholds["main_cost_1000"]
d = test.assign(score=test_scores, alert=test_scores >= t)

# Amount bands with edges from TRAIN, so test data doesn't shape them
edges = np.quantile(train["amount"], [0, 0.25, 0.5, 0.75, 1.0])
edges[0], edges[-1] = -np.inf, np.inf
d["amount_band"] = pd.cut(d["amount"], edges, labels=["Q1 low", "Q2", "Q3", "Q4 high"])
d["sender_balance"] = np.where(d["oldbalanceOrg"] > 0, "above 0", "zero")

def group_table(col):
    out = []
    for g, part in d.groupby(col, observed=True):
        y = part["isFraud"]
        a = part["alert"]
        out.append({
            col: g,
            "rows": len(part),
            "frauds": int(y.sum()),
            "recall": round((y & a).sum() / max(y.sum(), 1), 3),
            "false_alarm_rate": round((~y.astype(bool) & a).sum() / max((y == 0).sum(), 1), 4),
        })
    return pd.DataFrame(out)

display(group_table("type"))
display(group_table("amount_band"))
display(group_table("sender_balance"))

,type,rows,frauds,recall,false_alarm_rate
0,CASH_OUT,307909,295,0.875,0.0156
1,TRANSFER,73304,295,1.000,0.0111


,amount_band,rows,frauds,recall,false_alarm_rate
0,Q1 low,93917,95,0.853,0.0204
1,Q2,93879,66,0.788,0.0150
2,Q3,94452,84,0.893,0.0138
3,Q4 high,98965,345,1.000,0.0102


,sender_balance,rows,frauds,recall,false_alarm_rate
0,above 0,199974,589,0.939,0.0281
1,zero,181239,1,0.000,0.0001


In [10]:
t = thresholds["main_cost_1000"]
st = stress.assign(score=stress_scores)
st["block"] = ((st["step"] - 1) // 100) * 100 + 1
pd.DataFrame([
    {"block": b, "fraud_rate_%": round(part["isFraud"].mean() * 100, 2),
     **summary(part, part["score"].values, t)}
    for b, part in st.groupby("block")
])

,block,fraud_rate_%,alerts,alerts_per_hour,frauds,caught,precision,recall,missed_loss,total_cost,no_alert_cost
0,401,0.93,2873,28.7,1084,1041,0.362,0.960,4598611,7471611,1732234613
1,501,1.30,2297,23.0,1052,998,0.434,0.949,5979446,8276446,1382297405
2,601,2.90,1718,17.2,1110,1061,0.618,0.956,4268936,5986936,1787572679
3,701,9.39,608,14.1,490,466,0.766,0.951,2280954,2888954,870867669
